In [ ]:
import sys
import pickle
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('../')
from utils.shap_utils import shap_dotplot_by_risk, get_vals, create_dataframe, plot_shaps_comparison


# Get the data

In [ ]:
## Fill in your own settings here!!

# Insert your own WSI annotations here
wsi_annotations = ['Tumor', '--', 'Tissue edge: stroma > tumor', 'Tumor', 'Adipose > stroma', '--', 'Stroma > adipose', 'Adipose > stroma', 'Tumor: solid pattern, abundant cytoplasm', 'Tumor >> adipose', 'Stroma > tumor > immune cells', 'Tumor > stroma > immune cells', 'Normal ducts/lobules', 'Stroma > adipose', 'Edge stainings, blood vessels', 'Stroma >  immune cells']

# Data type
type = 'brca'

# Fold
fold = 2

# Name of experiment/model
exp_name = "DIMAFx"

In [ ]:
# Loading pathways names and initializing cluster identities
hallmarks = pd.read_csv('../data/data_files/hallmarks_signatures.csv')
hallmarks = np.array([' '.join(x[9:].split('_')) for x in hallmarks.columns])
hallmarks_ids = np.array([f'R{i}' for i, item in enumerate (hallmarks)])
hallmarks_names = np.array([f'R{i}: {item}' for i, item in enumerate (hallmarks)])

cluster_ids = np.array([f'W{i}' for i, item in enumerate (wsi_annotations)])
cluster_names = np.array([f'W{i}: {item}' for i, item in enumerate (wsi_annotations)])

In [ ]:
# Obtain shap values
shap_results_fold_dir = f'../results/dss_survival_{type}/{exp_name}/Fold_{fold}/post_training/shap/post_attn/shap_all_test.pkl'
shap_dict = pickle.load(open(shap_results_fold_dir, 'rb'))     
shap_values = shap_dict['shap values']

# Obtain the sorted features list corresponding to the highest to lowest shap of each of the representations 
z_hh = get_vals(shap_values[:, -16:, :], axis=2) # from Z^p_hh,i
z_gg = get_vals(shap_values[:, :50, :], axis=2) # from Z^p_gg,i
z_hg = get_vals(shap_values[:, 50:100, :], axis=2) # from Z^p_hg,i
z_gh = get_vals(shap_values[:, 100:116, :], axis=2) # from Z^p_gh,i


# Obtain risks
data_results_path = f"../results/dss_survival_{type}/{exp_name}/Fold_{fold}/post_training/predicted_risk_scores_test.pkl"
scores_results = pickle.load(open(data_results_path, 'rb'))   

In [ ]:
df_rna = create_dataframe(z_gg, z_hg, hallmarks_names)
df_wsi = create_dataframe(z_hh, z_gh, cluster_names)
df_total = create_dataframe(np.concatenate((z_gg, z_hh)), np.concatenate((z_hg, z_gh)), np.concatenate((hallmarks_names, cluster_names)))
df_total = df_total/ df_total.sum().sum()

In [ ]:
# Create gradients for reddish and bluish tones
reds = plt.cm.Reds(np.linspace(0.4, 1, 16))   # lighter to darker reds
blues = plt.cm.Blues(np.linspace(0.4, 1, 50)) # lighter to darker blues

colors = list(blues) + list(reds)

plot_shaps_comparison(
    df_total,
    label_features=["R48", "R8", "R32", "R12", "W4", "W8", "W15","W7"], 
    colors=colors,
    ylim=0.015,
    with_legend=False
)

# Plot shap plot vs risk score

In [ ]:
z_hh_all = np.sum(shap_values[:, -16:, :], axis=2)# from Z^p_hh,i
z_gg_all = np.sum(shap_values[:, :50, :], axis=2) # from Z^p_gg,i
z_hg_all = np.sum(shap_values[:, 50:100, :], axis=2) # from Z^p_hg,i
z_gh_all = np.sum(shap_values[:, 100:116, :], axis=2) # from Z^p_gh,i

In [ ]:
z_hh_all_feats = [f"Specific {name}" for name in cluster_names]
z_gg_all_feats = [f"Specific {name}" for name in hallmarks_names]
z_hg_all_feats = [f"Shared {name}" for name in hallmarks_names]
z_gh_all_feats = [f"Shared {name}" for name in cluster_names]

In [ ]:
z_all = np.hstack((z_hh_all, z_gg_all, z_hg_all, z_gh_all))
z_all_feats = z_hh_all_feats + z_gg_all_feats + z_hg_all_feats + z_gh_all_feats


In [ ]:
shap_dotplot_by_risk(z_all, z_all_feats, scores_results['Risk scores'])

# Plot SHAP of single sample

In [ ]:
def get_plot_single_sample(sample, shap_dict, all_feats_names, max_display=15):
    """ Plot SHAP values for a single sample."""
    # Get shap values of this sample
    shap_values = shap_dict['shap values']
    sample_index = shap_dict['Samples'].index(sample)
    shap_of_one_sample = np.expand_dims(np.sum(shap_values[sample_index], axis=-1),axis=0)

    # Plot shap
    sample_expl = shap.Explanation(values=shap_of_one_sample, feature_names=all_feats_names)
    shap.plots.bar(sample_expl[0], max_display=max_display)
    plt.show()

In [ ]:
# We can also plot the local multimodal feature importance plot of one sample
# Take a test id of the specified fold
id = 'TCGA-3C-AALK'
all_feats_names =  z_gg_all_feats + z_hg_all_feats + z_gh_all_feats + z_hh_all_feats
get_plot_single_sample(id, shap_dict, all_feats_names)